# Bedrock Converse API - PDFパーシングプロンプト検証ノートブック

ナレッジベースのFMパーサー用プロンプト（v3）を、Converse APIで単独テストするためのノートブックです。

## 前提条件
- AWS認証情報が設定済み（IAMロール、プロファイル、または環境変数）
- `anthropic.claude-3-5-sonnet-20241022-v2:0` へのモデルアクセスが有効
- テスト対象のPDFファイルが用意されていること

## 1. セットアップ

In [1]:
import boto3
import json
import os
import re
from pathlib import Path
from IPython.display import display, Markdown, HTML

## 2. 設定パラメータ

環境に合わせて変更してください。

In [2]:
# === 設定 ===

# リージョン
REGION = "us-west-2"

# モデルID
MODEL_ID = "anthropic.claude-3-5-sonnet-20241022-v2:0"

# テスト対象PDFファイルのパス
PDF_FILE_PATH = "./test_document.pdf"

# 推論パラメータ
MAX_TOKENS = 4096
TEMPERATURE = 0.0

# 結果出力先ディレクトリ
OUTPUT_DIR = "./output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 3. パーシングプロンプト定義

デフォルトプロンプト（英語）と日本語微修正版（v3）を定義します。  
比較検証のために両方を用意しています。

In [3]:
# デフォルトプロンプト（英語・ナレッジベースのFMパーサーのデフォルト）
PROMPT_DEFAULT_EN = """Extract the content from an image page and output in Markdown syntax. Enclose the content in the <markdown></markdown> tag and do not use code blocks. If the image is empty then output a <markdown></markdown> without anything in it.
Follow these steps:
1. Examine the provided page carefully.
2. Identify all elements present in the page, including headers, body text, footnotes, tables, images, captions, and page numbers, etc.
3. Use markdown syntax to format your output:
    - Headings: # for main, ## for sections, ### for subsections, etc.
    - Lists: * or - for bulleted, 1. 2. 3. for numbered
    - Do not repeat yourself
4. If the element is an image (not table)
    - If the information in the image can be represented by a table, generate the table containing the information of the image
    - Otherwise provide a detailed description about the information in image
    - Classify the element as one of: Chart, Diagram, Logo, Icon, Natural Image, Screenshot, Other. Enclose the class in <figure_type></figure_type>
    - Enclose <figure_type></figure_type>, the table or description, and the figure title or caption (if available), in <figure></figure> tags
    - Do not transcribe text in the image after providing the table or description
5. If the element is a table
    - Create a markdown table, ensuring every row has the same number of columns
    - Maintain cell alignment as closely as possible
    - Do not split a table into multiple tables
    - If a merged cell spans multiple rows or columns, place the text in the top-left cell and output ' ' for other
    - Use | for column separators, |-|-| for header row separators
    - If a cell has multiple items, list them in separate rows
    - If the table contains sub-headers, separate the sub-headers from the headers in another row
6. If the element is a paragraph
    - Transcribe each text element precisely as it appears
7. If the element is a header, footer, footnote, page number
    - Transcribe each text element precisely as it appears"""

In [4]:
# 日本語微修正版プロンプト（v3）
PROMPT_JA = """画像ページからコンテンツを抽出し、Markdown構文で出力してください。コンテンツは<markdown></markdown>タグで囲み、コードブロック（```）は使用しないでください。画像にコンテンツが含まれない（空である）場合は、中身のない<markdown></markdown>を出力してください。

実行手順：
ページの精査: 提供されたページを注意深く確認してください。

要素の識別: ヘッダー、本文テキスト、脚注、テーブル、画像、キャプション、ページ番号など、ページに存在するすべての要素を識別してください。

Markdown構文の使用:

見出し：メインは #、セクションは ##、サブセクションは ### など

リスト：箇条書きは * または -、番号付きは 1. 2. 3.

同じ内容を繰り返さないこと。

画像（テーブル以外）の処理:

画像内の情報がテーブルで表現できる場合は、テーブル形式で出力してください。それ以外の場合は、日本語で詳細な説明を記述してください。いずれか一方のみを出力し、両方を出力しないでください。

要素を次のいずれかに分類してください：Chart, Diagram, Logo, Icon, Natural Image, Screenshot, Other。分類は<figure_type></figure_type>で囲んでください。

<figure_type></figure_type>、テーブルまたは説明、および図のタイトルやキャプション（ある場合）を<figure></figure>タグで囲んでください。

テーブルまたは説明を提供した後に、画像内のテキストを重複して転記しないでください。

テーブルの処理:

Markdownテーブルを作成し、すべての行が同じ列数になるようにしてください。

セルの配置をできる限り忠実に維持してください。

テーブルを複数のテーブルに分割しないでください。

結合セルが複数の行または列にまたがる場合、テキストを左上のセルに配置し、他のセルには ' ' を出力してください。

列の区切りには | を使用し、ヘッダー行の区切りには |-|-| を使用してください。

セルに複数の項目がある場合は、別々の行にリストしてください。

テーブルにサブヘッダーが含まれる場合は、サブヘッダーをヘッダーから別の行に分離してください。

「〃」や「同上」などの表記は、そのまま転記してください。

段落の処理:

各テキスト要素を表示されている通りに正確に転記してください。

段落の途中で不自然に改行されている場合は、文脈がつながるように1つの連続した文章として整形してください。ただし、意図的な箇条書きや改行は保持してください。

共通要素（ヘッダー等）の処理:

ヘッダー、フッター、脚注、ページ番号は、表示されている通りに正確に転記してください。

日本語文書における追加ルール：
文字の維持: 漢字・ひらがな・カタカナ・英数字・特殊記号（①、②、※、■など）はすべて原文の通りに転記し、勝手に変換や省略をしないでください。

全角・半角の保持: 全角文字（例：１００ｍｍ）と半角文字（例：100mm）は原文の表記をそのまま保持してください。

句読点・記号: 日本語の句読点（。、）、括弧（「」『』（））、中黒（・）などはそのまま保持し、西洋の句読点（.や,）に置き換えないでください。

ルビ（ふりがな）: ルビが漢字の上または横に表示されている場合は、漢字の直後に括弧で記載してください。例：漢字(かんじ)

読み順の維持: 縦書き（縦組み）のレイアウトは、読み順を正しく保ちながら横書きのMarkdownに変換してください。

非翻訳: 日本語と英語が混在するテキストは、それぞれの言語をそのまま維持し、翻訳しないでください。

出力例：
以下は出力フォーマットの参考例です。

--- 例1: チャート画像（テーブルで表現可能な場合） ---
<markdown>
## 年次報告書

### 財務ハイライト

<figure>
<figure_type>Chart</figure_type>
図3: 年間売上の推移（百万ドル）
| 年 | 売上（百万ドル） |
|-|-|
| 2018 | $12M |
| 2019 | $18M |
| 2020 | $8M |
| 2021 | $22M |
</figure>

売上: $40M

利益: $12M

EPS: $1.25
</markdown>

--- 例2: ロゴ画像とテーブルが混在するページ ---
<markdown>
<figure>
<figure_type>Logo</figure_type>
Apple Inc.のロゴ。
</figure>

## キャッシュフロー計算書

| | 12月31日に終了した年度 | |
| | 2021 | 2022 |
|-|-|-|
| 以下により提供（使用）された現金: | | |
| 営業活動 | $ 46,327 | $ 46,752 |
| 投資活動 | (58,154) | (37,601) |
| 財務活動 | 6,291 | 9,718 |
</markdown>"""

In [5]:
# 使用するプロンプトを選択（切り替えて比較可能）
ACTIVE_PROMPT = PROMPT_JA  # PROMPT_DEFAULT_EN または PROMPT_JA

prompt_name = "日本語微修正版 v3" if ACTIVE_PROMPT == PROMPT_JA else "デフォルト（英語）"
print(f"選択中のプロンプト: {prompt_name}")
print(f"プロンプト文字数: {len(ACTIVE_PROMPT):,}")

選択中のプロンプト: 日本語微修正版 v3
プロンプト文字数: 2,214


## 4. Bedrockクライアント初期化

In [6]:
bedrock_runtime = boto3.client(
    service_name="bedrock-runtime",
    region_name=REGION
)

print(f"リージョン: {REGION}")
print(f"モデル: {MODEL_ID}")

リージョン: us-west-2
モデル: anthropic.claude-3-5-sonnet-20241022-v2:0


## 5. PDF読み込みとConverse API呼び出し関数

In [7]:
def read_pdf(file_path: str) -> bytes:
    """PDFファイルをバイト列として読み込む"""
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f"ファイルが見つかりません: {file_path}")
    
    size_mb = path.stat().st_size / (1024 * 1024)
    print(f"ファイル: {path.name}")
    print(f"サイズ: {size_mb:.2f} MB")
    
    if size_mb > 4.5:
        raise ValueError(f"ファイルサイズが4.5MBの上限を超えています: {size_mb:.2f} MB")
    
    with open(path, "rb") as f:
        return f.read()


def parse_pdf_with_converse(
    client,
    model_id: str,
    pdf_bytes: bytes,
    prompt: str,
    doc_name: str = "document",
    max_tokens: int = 4096,
    temperature: float = 0.0,
) -> dict:
    """Converse APIでPDFをパーシングする"""
    
    message = {
        "role": "user",
        "content": [
            {
                "document": {
                    "name": doc_name,
                    "format": "pdf",
                    "source": {
                        "bytes": pdf_bytes
                    }
                }
            },
            {
                "text": prompt
            }
        ]
    }
    
    response = client.converse(
        modelId=model_id,
        messages=[message],
        inferenceConfig={
            "maxTokens": max_tokens,
            "temperature": temperature
        }
    )
    
    return response


def extract_result(response: dict) -> str:
    """レスポンスからテキスト結果を抽出する"""
    output_message = response["output"]["message"]
    result_text = ""
    for block in output_message["content"]:
        if "text" in block:
            result_text += block["text"]
    return result_text


def extract_markdown_content(text: str) -> str:
    """<markdown>タグ内のコンテンツを抽出する"""
    pattern = r"<markdown>(.*?)</markdown>"
    matches = re.findall(pattern, text, re.DOTALL)
    if matches:
        return "\n\n---\n\n".join(m.strip() for m in matches)
    # タグがない場合はそのまま返す
    return text


def print_usage(response: dict):
    """トークン使用量を表示する"""
    usage = response["usage"]
    print(f"入力トークン数:  {usage['inputTokens']:,}")
    print(f"出力トークン数:  {usage['outputTokens']:,}")
    print(f"合計トークン数:  {usage['totalTokens']:,}")
    print(f"終了理由:        {response['stopReason']}")

## 6. パーシング実行

In [8]:
# PDFファイルを読み込み
pdf_bytes = read_pdf(PDF_FILE_PATH)

# Converse APIでパーシング実行
print(f"\nパーシング実行中...")
print(f"使用プロンプト: {prompt_name}")
print("-" * 50)

response = parse_pdf_with_converse(
    client=bedrock_runtime,
    model_id=MODEL_ID,
    pdf_bytes=pdf_bytes,
    prompt=ACTIVE_PROMPT,
    doc_name=Path(PDF_FILE_PATH).stem,
    max_tokens=MAX_TOKENS,
    temperature=TEMPERATURE,
)

# トークン使用量を表示
print_usage(response)

# 結果を抽出
result_text = extract_result(response)

ファイル: test_document.pdf
サイズ: 0.09 MB

パーシング実行中...
使用プロンプト: 日本語微修正版 v3
--------------------------------------------------
入力トークン数:  8,072
出力トークン数:  1,677
合計トークン数:  9,749
終了理由:        end_turn


## 7. 結果表示（Raw テキスト）

In [9]:
print(result_text)

<markdown>
# NA-VX900CL 取扱説明書

## 安全上のご注意

この取扱説明書には、事故を防ぐための重要な注意事項と製品の取り扱いかたを示しています。
お読みになったあとは、いつでも見られるところに必ず保管してください。

※ ご使用の前に必ずお読みください

1. 設置場所は水平で安定した場所を選んでください。
2. 電源プラグはアース付きの専用コンセント（AC100V・15A以上）に差し込んでください。
3. 給水ホースは確実に接続し、水漏れがないか確認してください。

※ 蛇口の水圧が0.03〜0.8MPaの範囲であることを確認してください。

■ 禁止事項：本体の上に物を載せないでください。

## 製品仕様（概要）

| 項目 | 仕様 |
|-|-|
| 外形寸法 | 幅639mm × 奥行722mm × 高さ1058mm |
| 質量 | 約79kg |
| 消費電力 | 洗濯時 200W ／ 乾燥時 1,190W |
| 運転音 | 洗い 32dB ／ 脱水 41dB ／ 乾燥 46dB |
| 対応電圧 | ＡＣ１００Ｖ ５０／６０Ｈｚ |

※1 JIS C 9811に基づく測定値です。実際の使用状況により異なります。
</markdown>

<markdown>
## 洗濯コース一覧

| コース名 | 洗濯容量 | 所要時間（目安） | 使用水量（目安） | 対応衣類 |
|-|-|-|-|-|
| 標準コース | | | | |
| おまかせ | 12kg | 約43分 | 約78L | 普段着全般 |
| 〃 | 6kg | 約35分 | 約57L | 〃 |
| おしゃれ着 | 3kg | 約40分 | 約52L | デリケート素材 |
| 乾燥コース | | | | |
| ヒートポンプ乾燥 | 6kg | 約98分 | — | 綿・化繊 |
| 省エネ乾燥 | 6kg | 約170分 | — | 綿・化繊 |
| スチーム乾燥 | 1kg | 約30分 | — | スーツ・制服 |
| 特殊コース | | | | |
| 温水泡洗浄W | 3kg | 約55分 | 約63L | 黄ばみ・皮脂汚れ |
| ダニバスター | 3kg | 約195分 | 約82L | 毛布・シーツ |
| 槽洗浄 | — | 約1

## 8. 結果表示（Markdownレンダリング）

`<markdown>` タグ内のコンテンツをMarkdownとしてレンダリングします。

In [10]:
markdown_content = extract_markdown_content(result_text)
display(Markdown(markdown_content))

# NA-VX900CL 取扱説明書

## 安全上のご注意

この取扱説明書には、事故を防ぐための重要な注意事項と製品の取り扱いかたを示しています。
お読みになったあとは、いつでも見られるところに必ず保管してください。

※ ご使用の前に必ずお読みください

1. 設置場所は水平で安定した場所を選んでください。
2. 電源プラグはアース付きの専用コンセント（AC100V・15A以上）に差し込んでください。
3. 給水ホースは確実に接続し、水漏れがないか確認してください。

※ 蛇口の水圧が0.03〜0.8MPaの範囲であることを確認してください。

■ 禁止事項：本体の上に物を載せないでください。

## 製品仕様（概要）

| 項目 | 仕様 |
|-|-|
| 外形寸法 | 幅639mm × 奥行722mm × 高さ1058mm |
| 質量 | 約79kg |
| 消費電力 | 洗濯時 200W ／ 乾燥時 1,190W |
| 運転音 | 洗い 32dB ／ 脱水 41dB ／ 乾燥 46dB |
| 対応電圧 | ＡＣ１００Ｖ ５０／６０Ｈｚ |

※1 JIS C 9811に基づく測定値です。実際の使用状況により異なります。

---

## 洗濯コース一覧

| コース名 | 洗濯容量 | 所要時間（目安） | 使用水量（目安） | 対応衣類 |
|-|-|-|-|-|
| 標準コース | | | | |
| おまかせ | 12kg | 約43分 | 約78L | 普段着全般 |
| 〃 | 6kg | 約35分 | 約57L | 〃 |
| おしゃれ着 | 3kg | 約40分 | 約52L | デリケート素材 |
| 乾燥コース | | | | |
| ヒートポンプ乾燥 | 6kg | 約98分 | — | 綿・化繊 |
| 省エネ乾燥 | 6kg | 約170分 | — | 綿・化繊 |
| スチーム乾燥 | 1kg | 約30分 | — | スーツ・制服 |
| 特殊コース | | | | |
| 温水泡洗浄W | 3kg | 約55分 | 約63L | 黄ばみ・皮脂汚れ |
| ダニバスター | 3kg | 約195分 | 約82L | 毛布・シーツ |
| 槽洗浄 | — | 約11時間 | 約17L | — |

※ 所要時間・使用水量は目安です。衣類の量・種類・水温により変動します。
※ 「〃」は上の行と同じ内容を示します。
※ 温水泡洗浄Wでは、約40℃の温水を使用します。

## エラーコード一覧

| エラーコード | 内容と対処方法 |
|-|-|
| U11 | 排水できません。排水口・排水ホースの詰まりを確認してください。 |
| U13 | ドアロックの異常です。ドアを閉め直してください。 |
| U14 | 給水できません。蛇口が開いているか確認してください。 |
| H27 | 温度センサーの異常です。電源プラグを抜き、販売店にご相談ください。 |
| H59 | モーターの異常です。電源プラグを抜き、販売店にご相談ください。 |

---

## 省エネ性能比較

<figure>
<figure_type>Chart</figure_type>
図1: 年間消費電力量の推移（2020年〜2024年モデル比較）
| 年度 | モデル | 消費電力量(kWh) |
|-|-|-|
| 2020年 | NA-VX800B | 590 |
| 2021年 | NA-VX800C | 520 |
| 2022年 | NA-VX900A | 470 |
| 2023年 | NA-VX900B | 430 |
| 2024年 | NA-VX900C | 380 |
</figure>

2020年モデル（NA-VX800B）と比較して、2024年モデル（NA-VX900C）では年間消費電力量を約35.6%削減しました。ヒートポンプ乾燥システムの改良とAIエコナビによる最適運転制御が主な要因です。

## 主な省エネ技術

* ヒートポンプ（Heat Pump）乾燥方式による高効率な熱回収
* AIエコナビ（AI ECONAVI）による衣類量・素材の自動検知
* インバーター（Inverter）制御モーターの高効率化
* 温水泡洗浄W（Warm Water Bubble Wash W）による洗浄力向上

※1 年間消費電力量はJIS C 9811:2015に基づく算出値です。
※2 実際の消費電力量は使用頻度・設定・環境温度等により異なります。
※3 AIエコナビはPanasonic独自のAI省エネ技術のブランド名です。

## 9. 結果をファイルに保存

In [11]:
pdf_stem = Path(PDF_FILE_PATH).stem
prompt_label = "ja_v3" if ACTIVE_PROMPT == PROMPT_JA else "en_default"

# Rawレスポンスを保存
raw_path = os.path.join(OUTPUT_DIR, f"{pdf_stem}_{prompt_label}_raw.txt")
with open(raw_path, "w", encoding="utf-8") as f:
    f.write(result_text)
print(f"Rawテキスト保存: {raw_path}")

# Markdownコンテンツを保存
md_path = os.path.join(OUTPUT_DIR, f"{pdf_stem}_{prompt_label}_parsed.md")
with open(md_path, "w", encoding="utf-8") as f:
    f.write(markdown_content)
print(f"Markdown保存:    {md_path}")

# トークン使用量をJSON保存
usage_path = os.path.join(OUTPUT_DIR, f"{pdf_stem}_{prompt_label}_usage.json")
usage_data = {
    "model_id": MODEL_ID,
    "prompt_type": prompt_label,
    "pdf_file": PDF_FILE_PATH,
    "input_tokens": response["usage"]["inputTokens"],
    "output_tokens": response["usage"]["outputTokens"],
    "total_tokens": response["usage"]["totalTokens"],
    "stop_reason": response["stopReason"]
}
with open(usage_path, "w", encoding="utf-8") as f:
    json.dump(usage_data, f, ensure_ascii=False, indent=2)
print(f"使用量JSON保存:  {usage_path}")

Rawテキスト保存: ./output/test_document_ja_v3_raw.txt
Markdown保存:    ./output/test_document_ja_v3_parsed.md
使用量JSON保存:  ./output/test_document_ja_v3_usage.json


## 10. 英語 vs 日本語プロンプト 比較実行

同じPDFに対して両方のプロンプトで実行し、結果を比較します。

In [12]:
def run_comparison(client, model_id, pdf_path, max_tokens=4096, temperature=0.0):
    """英語・日本語プロンプトの比較実行"""
    pdf_bytes = read_pdf(pdf_path)
    pdf_stem = Path(pdf_path).stem
    
    results = {}
    prompts = {
        "en_default": ("デフォルト（英語）", PROMPT_DEFAULT_EN),
        "ja_v3": ("日本語微修正版 v3", PROMPT_JA),
    }
    
    for key, (label, prompt) in prompts.items():
        print(f"\n{'='*60}")
        print(f"実行中: {label}")
        print(f"{'='*60}")
        
        resp = parse_pdf_with_converse(
            client=client,
            model_id=model_id,
            pdf_bytes=pdf_bytes,
            prompt=prompt,
            doc_name=pdf_stem,
            max_tokens=max_tokens,
            temperature=temperature,
        )
        
        print_usage(resp)
        text = extract_result(resp)
        
        results[key] = {
            "label": label,
            "text": text,
            "markdown": extract_markdown_content(text),
            "usage": resp["usage"],
            "stop_reason": resp["stopReason"]
        }
        
        # ファイル保存
        md_path = os.path.join(OUTPUT_DIR, f"{pdf_stem}_{key}_parsed.md")
        with open(md_path, "w", encoding="utf-8") as f:
            f.write(results[key]["markdown"])
        print(f"保存: {md_path}")
    
    return results


# 比較を実行する場合は以下のコメントを解除
# comparison = run_comparison(bedrock_runtime, MODEL_ID, PDF_FILE_PATH)

In [13]:
# 比較結果の並列表示（上のセルで比較実行した場合）
# for key, data in comparison.items():
#     print(f"\n{'='*60}")
#     print(f"{data['label']}")
#     print(f"入力: {data['usage']['inputTokens']:,} / 出力: {data['usage']['outputTokens']:,}")
#     print(f"{'='*60}")
#     display(Markdown(data["markdown"][:3000] + "\n\n...(以下省略)"))